# Phase 2: Evidential Uncertainty & Cross-Layer Agreement for Semi-Supervised Building Segmentation

**Core Architecture:** DABLCNet (ViT@512 + ASPP + Edge-Guided Decoder)

**Novel Contributions (Journal-Level):**
1. **EDL-Conf (Evidential Dirichlet Head)** — Principled epistemic uncertainty via Subjective Logic (replaces sigmoid confidence)
2. **CLAAM / MLAA (Cross-Layer Attention Agreement Map)** — Per-layer ViT segmentation heads measure encoder-internal agreement
3. **DU-Gate (7-Gate Multi-Uncertainty Pseudo-Label Acceptance)** — Fuses output-level (EDL), encoder-level (CLAAM), and training-dynamics-level (TVR) uncertainty

**Additional Novel Ideas:**
4. **CLAC-Loss** — Cross-layer spatial consistency regularization between ViT layers
5. **TVR (Temporal Voting across Rotations)** — Pseudo-label stability across label rotation cycles
6. **GOP (Gradient Orientation Prior)** — Rectilinear boundary prior for building-specific validation
7. **CAFCG (Frequency Consistency Gate)** — Low-freq/high-freq agreement as quality control
8. **SDR (Signed Distance Regression Head)** — Predicts signed distance to building boundaries for sub-pixel precision

**Existing Components:** DABL-C loss, HH wavelet edge branch, triple-gate (geometry + confidence + edge), 3-stage training, label rotation

## Phase-2 Coding Steps

1) Configure paths, label fraction, low-epoch settings, + **novel contribution ablation switches**
2) Build model: ViT encoder, edge branch w/ HH wavelet, edge-guided decoder
3) **NEW: EvidentialHead (EDL-Conf)** — Dirichlet α → epistemic uncertainty (Contribution 1)
4) **NEW: CLAAM/MLAA** — Per-layer ViT seg heads + agreement map (Contribution 2)
5) **NEW: CLAC-Loss** — Cross-layer spatial consistency regularization (Idea 2)
6) **NEW: TVR** — Temporal voting buffer across label rotations (Idea 3)
7) **NEW: GOP** — Gradient orientation prior gate (Idea 4)
8) **NEW: CAFCG** — Frequency consistency gate (Idea 5)
9) DABL-C loss + EDL loss + CLAAM consistency loss + CLAC regularization
10) **7-gate multi-uncertainty pseudo-label acceptance** (replaces triple-gate)
11) 3-stage training: GT → pseudo-gen (7-gate) → finetune
12) Full metrics + ablation switches (all contributions independently toggleable)
13) **Uncertainty visualization** — EDL + CLAAM + confidence maps (paper figures)

In [ ]:
# Cell 3
import math
import os
import glob
import random
from dataclasses import dataclass, field
from typing import Dict, Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm.auto import tqdm
import timm

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

@dataclass
class Config:
    # Dataset paths
    data_root: str = "/kaggle/input/datasets/sengulgs/whu-building-dataset/WHU"
    img_size: int = 512

    # Training
    label_frac: float = 0.1
    batch_size: int = 2
    epochs_stage1: int = 15
    epochs_stage2: int = 15
    epochs_stage3: int = 15
    warmup_freeze: int = 3

    # Learning rates
    lr_backbone: float = 1e-5
    lr_head: float = 3e-4
    lr_loss_weights: float = 5e-5    # Separate (slow) LR for DABL-C w_in/w_out

    # Pseudo-label thresholds
    thr_start: float = 0.65
    thr_end: float = 0.50

    # Loss weights
    loss_w_in: float = 1.0
    loss_w_out: float = 1.5
    loss_dice_w: float = 1.0
    boundary_weight: float = 2.5      # Increased: gap-aware loss needs more pressure
    conf_loss_weight: float = 0.3

    # Preview settings
    preview_every: int = 5
    preview_samples: int = 2
    pred_thr: float = 0.5

    # Early stopping + model saving
    patience: int = 12
    save_path: str = "best_model.pt"

    # EDL hyperparameters
    edl_loss_weight: float = 0.5
    edl_annealing_epochs: int = 10

    # CLAAM hyperparameters (reduced weight to prevent head collapse)
    claam_loss_weight: float = 0.1
    claam_diversity_weight: float = 0.05
    claam_thr_start: float = 0.85
    claam_thr_end: float = 0.70

    # CLAC hyperparameters
    clac_loss_weight: float = 0.2

    # TVR hyperparameters
    tvr_window: int = 5
    tvr_accept_ratio: float = 0.6

    # GOP hyperparameters
    gop_rectilinear_thr: float = 0.45

    # CAFCG hyperparameters
    cafcg_disagreement_thr: float = 0.3

    # SDR (Signed Distance Regression) hyperparameters
    sdr_loss_weight: float = 0.3      # Weight for SDF regression loss
    sdr_max_dist: float = 50.0        # Normalize SDF to [-1, +1] using this max distance (px)

    # Pseudo-label settings
    pseudo_ramp_start: float = 0.1    # Loss ramp: start at 10% weight
    pseudo_freeze_backbone: bool = True  # Freeze ViT during S2 to protect features
    pseudo_edge_erode_px: int = 5     # Erode pseudo-labels by N px to prevent merging

    # Cross-dataset evaluation
    inria_root: str = "/kaggle/input/inria-building/AerialImageDataset"
    cross_dataset_img_size: int = 512

cfg = Config()
device = get_device()
print("Device:", device)
print("All novel contributions enabled: EDL, CLAAM, CLAC, TVR, GOP, CAFCG, SDR")

In [ ]:
# Cell 4
def rgb_to_gray(x: torch.Tensor) -> torch.Tensor:
    # x: [B,3,H,W]
    r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
    return 0.2989 * r + 0.5870 * g + 0.1140 * b


def hh_wavelet_channel(x: torch.Tensor) -> torch.Tensor:
    # Simple HH (diagonal) Haar wavelet channel. Returns [B,1,H,W].
    gray = rgb_to_gray(x)
    kernel = torch.tensor([[1.0, -1.0], [-1.0, 1.0]], device=gray.device).view(1, 1, 2, 2)
    hh = F.conv2d(gray, kernel, stride=1, padding=1)
    hh = hh[:, :, :x.shape[-2], :x.shape[-1]]
    return hh


class EdgeBranch(nn.Module):
    def __init__(self, in_ch: int = 3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        return x


In [ ]:
# Cell 5
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class SpatialEncoder(nn.Module):
    """Lightweight CNN for multi-scale spatial features → decoder skip connections.
    Gives the decoder high-res boundary info that ViT's 32×32 bottleneck loses.
    Only ~160K params — negligible vs ViT's 86M."""
    def __init__(self):
        super().__init__()
        # 512 → 256, 32ch
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        # 256 → 128, 64ch
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))

    def forward(self, x):
        s1 = self.layer1(x)   # [B, 32, H/2, W/2]  = 256
        s2 = self.layer2(s1)  # [B, 64, H/4, W/4]  = 128
        return s1, s2


class ViTEncoder(nn.Module):
    """512-resolution ViT: 512/16 = 32×32 tokens (vs 14×14 at 224).
    Small buildings that were <1px at 14×14 now survive at 32×32.
    Pos embeddings interpolated from pretrained 224 weights by timm.
    Grad checkpointing saves ~50% activation VRAM for T4 16GB."""
    def __init__(self):
        super().__init__()
        if timm is None:
            raise RuntimeError("timm not available.")
        self.vit = timm.create_model(
            "vit_base_patch16_224", pretrained=True,
            num_classes=0, img_size=512)
        try:
            self.vit.set_grad_checkpointing(enable=True)
        except AttributeError:
            pass  # older timm — skip grad checkpointing
        self.patch_size = 16
        self.embed_dim = 768
        # Tap layers 3, 6, 9, 12 (0-indexed: 2, 5, 8, 11)
        self.hook_layers = [2, 5, 8, 11]
        self._features = {}
        self._register_hooks()

    def _register_hooks(self):
        """Register forward hooks on intermediate transformer blocks."""
        for idx in self.hook_layers:
            block = self.vit.blocks[idx]
            block.register_forward_hook(self._make_hook(idx))

    def _make_hook(self, idx):
        def hook_fn(module, input, output):
            self._features[idx] = output
        return hook_fn

    def forward(self, x: torch.Tensor):
        # ImageNet normalization (CRITICAL — ViT was trained on normalized images)
        mean = IMAGENET_MEAN.to(x.device)
        std = IMAGENET_STD.to(x.device)
        x = (x - mean) / std

        self._features = {}
        _ = self.vit.forward_features(x)

        b = x.shape[0]
        grid_h = x.shape[-2] // self.patch_size   # 32
        grid_w = x.shape[-1] // self.patch_size    # 32

        multi_feats = []
        for idx in self.hook_layers:
            feat = self._features[idx]             # [B, 1025, 768]
            if feat.shape[1] == grid_h * grid_w + 1:
                feat = feat[:, 1:, :]              # drop CLS
            feat_map = feat.transpose(1, 2).reshape(b, self.embed_dim, grid_h, grid_w)
            multi_feats.append(feat_map)

        return multi_feats  # [layer3@32, layer6@32, layer9@32, layer12@32]


class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling (DeepLabV3+ style).
    Captures multi-scale context at the bottleneck — buildings come in many sizes.
    Parallel dilated convolutions at rates 6, 12, 18 + global average pooling."""
    def __init__(self, in_ch, out_ch=256):
        super().__init__()
        self.conv1x1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.atrous6 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=6, dilation=6, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.atrous12 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=12, dilation=12, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.atrous18 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=18, dilation=18, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.ReLU(inplace=True))
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * 5, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Dropout(0.1))

    def forward(self, x):
        h, w = x.shape[-2:]
        f1 = self.conv1x1(x)
        f2 = self.atrous6(x)
        f3 = self.atrous12(x)
        f4 = self.atrous18(x)
        f5 = self.gap(x)
        f5 = F.interpolate(f5, size=(h, w), mode="bilinear", align_corners=False)
        out = torch.cat([f1, f2, f3, f4, f5], dim=1)
        return self.project(out)


class MultiLayerFusion(nn.Module):
    """Fuse 4 ViT layers (each 768-ch @ 32×32) into a single feature map.
    Projects each layer to 256-ch, concatenates, then compresses to out_ch."""
    def __init__(self, embed_dim=768, out_ch=768):
        super().__init__()
        self.proj3  = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.proj6  = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.proj9  = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.proj12 = nn.Sequential(nn.Conv2d(embed_dim, 256, 1), nn.BatchNorm2d(256), nn.ReLU(True))
        self.fuse = nn.Sequential(
            nn.Conv2d(256 * 4, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))

    def forward(self, multi_feats):
        p3  = self.proj3(multi_feats[0])
        p6  = self.proj6(multi_feats[1])
        p9  = self.proj9(multi_feats[2])
        p12 = self.proj12(multi_feats[3])
        return self.fuse(torch.cat([p3, p6, p9, p12], dim=1))


class EdgeGuidedDecoder(nn.Module):
    """Decoder for 32×32 bottleneck → 512. All learned upsampling — no bilinear blur.
    32→64→128(+skip@128)→256(+skip@256)→512 with edge fusion."""
    def __init__(self, in_ch: int = 256):
        super().__init__()
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.skip_fuse2 = nn.Sequential(
            nn.Conv2d(192, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.skip_fuse3 = nn.Sequential(
            nn.Conv2d(96, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up4 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.refine = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(
            nn.Conv2d(32 + 1, 32, 3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.out_conv = nn.Conv2d(32, 1, 1)

    def _decode(self, feat, out_size, skips=None):
        """Shared decode path: 32→64→128→256→512."""
        x = self.up1(feat)
        x = self.up2(x)
        if skips is not None:
            s2 = F.interpolate(skips[1], size=x.shape[-2:],
                               mode="bilinear", align_corners=False)
            x = self.skip_fuse2(torch.cat([x, s2], dim=1))
        x = self.up3(x)
        if skips is not None:
            s1 = F.interpolate(skips[0], size=x.shape[-2:],
                               mode="bilinear", align_corners=False)
            x = self.skip_fuse3(torch.cat([x, s1], dim=1))
        x = self.up4(x)
        if x.shape[-2] != out_size[0] or x.shape[-1] != out_size[1]:
            x = F.interpolate(x, size=out_size, mode="nearest")
        x = self.refine(x)
        return x

    def forward(self, feat: torch.Tensor, edge: torch.Tensor,
                out_size: Tuple[int, int], skips=None) -> torch.Tensor:
        x = self._decode(feat, out_size, skips)
        edge_resized = F.interpolate(edge, size=out_size, mode="nearest")
        x = torch.cat([x, edge_resized], dim=1)
        x = self.fuse(x)
        return self.out_conv(x)

    def get_features(self, feat: torch.Tensor, out_size: Tuple[int, int],
                     skips=None) -> torch.Tensor:
        """Return 32-ch feature map for confidence head (no edge fusion)."""
        return self._decode(feat, out_size, skips)


class ConfidenceHead(nn.Module):
    """Original sigmoid confidence head (baseline / fallback when EDL disabled)."""
    def __init__(self, in_ch: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 16, 3, padding=1)
        self.conv3 = nn.Conv2d(16, 1, 1)
        self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        temp = torch.exp(self.log_temp) + 1e-6
        return torch.sigmoid(x / temp)


# ═══════════════════════════════════════════════════════════════════
#  CONTRIBUTION 1: Evidential Dirichlet Confidence Head (EDL-Conf)
# ═══════════════════════════════════════════════════════════════════

class EvidentialHead(nn.Module):
    """Outputs Dirichlet concentration params α = evidence + 1 per class.
    Epistemic uncertainty u = K/S where S = Σα, K = num_classes."""
    def __init__(self, in_ch: int = 32, num_classes: int = 2):
        super().__init__()
        self.K = num_classes
        self.conv1 = nn.Conv2d(in_ch, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 16, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.conv3 = nn.Conv2d(16, num_classes, 1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        evidence = F.softplus(self.conv3(x))
        alpha = evidence + 1.0
        S = alpha.sum(dim=1, keepdim=True)
        belief = evidence / S
        uncertainty = float(self.K) / S
        confidence = 1.0 - uncertainty
        return {
            "alpha": alpha,
            "belief": belief,
            "uncertainty": uncertainty,
            "confidence": confidence.clamp(0, 1),
            "strength": S,
        }


def kl_dirichlet(alpha: torch.Tensor, num_classes: int = 2) -> torch.Tensor:
    """KL divergence between Dir(α) and Dir(1,...,1)."""
    ones = torch.ones_like(alpha)
    S_alpha = alpha.sum(dim=1, keepdim=True)
    S_ones = ones.sum(dim=1, keepdim=True)
    kl = (torch.lgamma(S_alpha) - torch.lgamma(S_ones)
          - (torch.lgamma(alpha) - torch.lgamma(ones)).sum(dim=1, keepdim=True)
          + ((alpha - ones) * (torch.digamma(alpha) - torch.digamma(S_alpha))).sum(dim=1, keepdim=True))
    return kl.mean()


def edl_loss(alpha, target, epoch, total_epochs, annealing_epochs=10):
    """EDL loss = Bayes risk + annealed KL."""
    target_1h = torch.cat([1.0 - target, target], dim=1)
    S = alpha.sum(dim=1, keepdim=True)
    bayes_risk = (target_1h * (torch.digamma(S) - torch.digamma(alpha))).sum(dim=1)
    annealing_coef = min(1.0, epoch / max(1, annealing_epochs))
    alpha_tilde = target_1h + (1.0 - target_1h) * (alpha - 1.0) + 1.0
    kl = kl_dirichlet(alpha_tilde, num_classes=2)
    return bayes_risk.mean() + annealing_coef * kl


# ═══════════════════════════════════════════════════════════════════
#  CONTRIBUTION 2: Cross-Layer Attention Agreement Map (CLAAM / MLAA)
# ═══════════════════════════════════════════════════════════════════

class CLAAM(nn.Module):
    """Cross-Layer Attention Agreement Map.
    Per-layer ViT segmentation heads measure encoder-internal agreement."""
    def __init__(self, embed_dim: int = 768, num_layers: int = 4):
        super().__init__()
        self.num_layers = num_layers
        self.layer_heads = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(embed_dim, 64, 1, bias=False),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.Conv2d(64, 1, 1)
            ) for _ in range(num_layers)
        ])

    def forward(self, multi_feats: List[torch.Tensor],
                out_size: Tuple[int, int]) -> Dict[str, torch.Tensor]:
        layer_logits = []
        layer_probs = []
        for i, feat in enumerate(multi_feats):
            logit = self.layer_heads[i](feat)
            logit = F.interpolate(logit, size=out_size,
                                  mode='bilinear', align_corners=False)
            layer_logits.append(logit)
            layer_probs.append(torch.sigmoid(logit))

        stacked = torch.stack(layer_probs, dim=0)
        mean_pred = stacked.mean(dim=0)
        variance = stacked.var(dim=0)
        agreement = 1.0 - (variance / 0.25).clamp(0, 1)

        return {
            "agreement": agreement,
            "variance": variance,
            "layer_probs": layer_probs,
            "layer_logits": layer_logits,
            "mean_pred": mean_pred,
        }


def claam_consistency_loss(layer_logits, target):
    """Force ALL ViT layers to agree with GT."""
    loss = torch.tensor(0.0, device=target.device)
    for logit in layer_logits:
        loss = loss + F.binary_cross_entropy_with_logits(logit, target)
    return loss / len(layer_logits)


# ═══════════════════════════════════════════════════════════════════
#  IDEA 2: Cross-Layer Attention Consistency Loss (CLAC-Loss)
# ═══════════════════════════════════════════════════════════════════

def clac_loss(multi_feats: List[torch.Tensor]) -> torch.Tensor:
    """Penalizes spatial activation disagreement between adjacent ViT layers."""
    loss = torch.tensor(0.0, device=multi_feats[0].device)
    for i in range(len(multi_feats) - 1):
        act_i = multi_feats[i].mean(dim=1, keepdim=True)
        act_j = multi_feats[i + 1].mean(dim=1, keepdim=True)
        act_i = (act_i - act_i.amin(dim=(2, 3), keepdim=True)) / \
                (act_i.amax(dim=(2, 3), keepdim=True) - act_i.amin(dim=(2, 3), keepdim=True) + 1e-6)
        act_j = (act_j - act_j.amin(dim=(2, 3), keepdim=True)) / \
                (act_j.amax(dim=(2, 3), keepdim=True) - act_j.amin(dim=(2, 3), keepdim=True) + 1e-6)
        loss = loss + F.mse_loss(act_i, act_j)
    return loss / (len(multi_feats) - 1)


# ═══════════════════════════════════════════════════════════════════
#  IDEA 4: Gradient Orientation Prior (GOP)
# ═══════════════════════════════════════════════════════════════════

def gradient_orientation_gate(probs: torch.Tensor, thr: float = 0.5,
                              rectilinear_thr: float = 0.45) -> torch.Tensor:
    """Reject pseudo-label components whose boundary gradient orientations
    deviate from rectilinear (0°/90°) angles."""
    import cv2
    B = probs.shape[0]
    gate = torch.ones_like(probs, dtype=torch.bool)

    for b in range(B):
        mask_np = (probs[b, 0].detach().cpu().numpy() > thr).astype(np.uint8) * 255
        if mask_np.sum() == 0:
            continue

        prob_np = probs[b, 0].detach().cpu().numpy()
        gx = cv2.Sobel(prob_np, cv2.CV_64F, 1, 0, ksize=3)
        gy = cv2.Sobel(prob_np, cv2.CV_64F, 0, 1, ksize=3)
        mag = np.sqrt(gx**2 + gy**2)
        angle = np.degrees(np.arctan2(gy, gx)) % 180

        contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        reject_mask = np.zeros_like(mask_np, dtype=np.uint8)

        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 30:
                continue
            comp_mask = np.zeros_like(mask_np)
            cv2.drawContours(comp_mask, [cnt], -1, 255, thickness=cv2.FILLED)
            kernel = np.ones((3, 3), np.uint8)
            dilated = cv2.dilate(comp_mask, kernel, iterations=2)
            eroded = cv2.erode(comp_mask, kernel, iterations=1)
            boundary = ((dilated > 0) & ~(eroded > 0))

            boundary_mag = mag[boundary]
            boundary_angle = angle[boundary]
            if boundary_mag.size == 0:
                continue
            strong = boundary_mag > (boundary_mag.mean() + 1e-8)

            if strong.sum() < 5:
                continue

            angles_strong = boundary_angle[strong]
            rectilinear = (
                (angles_strong < 15) | (angles_strong > 165) |
                ((angles_strong > 75) & (angles_strong < 105))
            )
            rect_frac = rectilinear.sum() / len(angles_strong)

            if rect_frac < rectilinear_thr:
                cv2.drawContours(reject_mask, [cnt], -1, 255, thickness=cv2.FILLED)

        if reject_mask.any():
            reject_t = torch.from_numpy(reject_mask > 0).to(probs.device)
            gate[b, 0] = gate[b, 0] & (~reject_t)

    return gate


# ═══════════════════════════════════════════════════════════════════
#  IDEA 5a: Confidence-Aware Frequency Consistency Gate (CAFCG)
# ═══════════════════════════════════════════════════════════════════

def cafcg_gate(probs: torch.Tensor, disagreement_thr: float = 0.3) -> torch.Tensor:
    """Frequency consistency gate: rejects pixels where low-freq prediction
    is confident but high-freq shows noisy/diffuse edges."""
    B, C, H, W = probs.shape
    gate = torch.ones(B, C, H, W, dtype=torch.bool, device=probs.device)

    ksize = 9
    sigma = 2.0
    x_coord = torch.arange(ksize, device=probs.device).float() - ksize // 2
    gauss_1d = torch.exp(-x_coord**2 / (2 * sigma**2))
    gauss_1d = gauss_1d / gauss_1d.sum()
    gauss_2d = gauss_1d.unsqueeze(1) * gauss_1d.unsqueeze(0)
    gauss_2d = gauss_2d.view(1, 1, ksize, ksize)

    low_freq = F.conv2d(probs, gauss_2d, padding=ksize // 2)
    high_freq = (probs - low_freq).abs()

    low_confident = (low_freq > 0.5).float()
    freq_disagreement = low_confident * high_freq

    gate = freq_disagreement.squeeze(1).unsqueeze(1) <= disagreement_thr
    return gate


# ═══════════════════════════════════════════════════════════════════
#  IDEA 5b: Signed Distance Regression Head (SDR)
# ═══════════════════════════════════════════════════════════════════
# Predicts signed distance to nearest building boundary per pixel.
# Positive = inside building, negative = outside, 0 = on boundary.
# Richer than binary mask — model learns spatial relationship to edges.
# Directly attacks merging: gap pixels have negative SDF = strong separator.
# Novel: No weakly supervised building segmentation uses SDF head.
# ═══════════════════════════════════════════════════════════════════

class SignedDistanceHead(nn.Module):
    """Predicts normalized signed distance field (SDF) from decoder features.
    Output range: tanh → [-1, +1] where +1 = deep interior, -1 = far outside,
    0 = on building boundary."""
    def __init__(self, in_ch: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        return torch.tanh(self.conv2(x))  # [-1, +1]


def compute_sdf_target(mask: torch.Tensor, max_dist: float = 50.0) -> torch.Tensor:
    """Compute normalized signed distance field from binary GT mask.
    Args:
        mask: [B, 1, H, W] binary building mask
        max_dist: clip and normalize distances to [-1, +1]
    Returns:
        sdf: [B, 1, H, W] where +1 = deep inside, -1 = far outside, 0 = boundary
    """
    from scipy.ndimage import distance_transform_edt
    B = mask.shape[0]
    sdf_batch = torch.zeros_like(mask)
    for b in range(B):
        m = mask[b, 0].cpu().numpy().astype(np.float64)
        if m.sum() > 0:
            dist_in = distance_transform_edt(m)
        else:
            dist_in = np.zeros_like(m)
        if (1 - m).sum() > 0:
            dist_out = distance_transform_edt(1 - m)
        else:
            dist_out = np.zeros_like(m)
        sdf = dist_in - dist_out  # + inside, - outside, 0 on boundary
        sdf = np.clip(sdf / max_dist, -1.0, 1.0)
        sdf_batch[b, 0] = torch.from_numpy(sdf).float()
    return sdf_batch.to(mask.device)


# ═══════════════════════════════════════════════════════════════════
#  Updated DABLCNet: integrates EDL, CLAAM, SDR, and all novel modules
# ═══════════════════════════════════════════════════════════════════

class DABLCNet(nn.Module):
    """DABLCNet + Novel Contributions:
    - EvidentialHead (EDL-Conf): Dirichlet uncertainty → Contribution 1
    - CLAAM (MLAA): Cross-layer attention agreement → Contribution 2
    - SignedDistanceHead (SDR): SDF regression for boundary precision → Idea 5
    - HH Wavelet EdgeBranch, ASPP, EdgeGuidedDecoder, SpatialEncoder
    """
    def __init__(self):
        super().__init__()

        # ── Core architecture ──
        self.encoder = ViTEncoder()
        self.layer_fuse = MultiLayerFusion(768, out_ch=768)
        self.aspp = ASPP(in_ch=768, out_ch=256)
        self.spatial_enc = SpatialEncoder()
        self.edge_branch = EdgeBranch(in_ch=4)
        self.decoder = EdgeGuidedDecoder(in_ch=256)

        # ── CONTRIBUTION 1: Evidential Dirichlet head ──
        self.edl_head = EvidentialHead(in_ch=32, num_classes=2)

        # ── CONTRIBUTION 2: CLAAM / MLAA ──
        self.claam = CLAAM(embed_dim=768, num_layers=4)

        # ── IDEA 5: Signed Distance Regression head ──
        self.sdr_head = SignedDistanceHead(in_ch=32)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        # ── Edge branch (HH wavelet augmented) ──
        hh = hh_wavelet_channel(x)
        edge_in = torch.cat([x, hh], dim=1)
        edge_map = self.edge_branch(edge_in)

        # ── ViT encoder → multi-layer features ──
        multi_feats = self.encoder(x)

        # ── Fusion + ASPP + Decoder ──
        fused = self.layer_fuse(multi_feats)
        bottleneck = self.aspp(fused)
        skips = self.spatial_enc(x)
        logits = self.decoder(bottleneck, edge_map,
                              out_size=x.shape[-2:], skips=skips)

        # ── Decoder features for auxiliary heads ──
        dec_feat = self.decoder.get_features(bottleneck,
                                              out_size=x.shape[-2:], skips=skips)

        # ── CONTRIBUTION 1: EDL confidence ──
        edl_out = self.edl_head(dec_feat)
        conf_map = edl_out["confidence"]

        # ── CONTRIBUTION 2: CLAAM / MLAA ──
        claam_out = self.claam(multi_feats, out_size=x.shape[-2:])

        # ── IDEA 5: Signed Distance prediction ──
        sdf_pred = self.sdr_head(dec_feat)

        return {
            "logits": logits,
            "edge": edge_map,
            "conf": conf_map,
            "edl": edl_out,
            "claam": claam_out,
            "multi_feats": multi_feats,
            "sdf_pred": sdf_pred,
        }

In [ ]:
# Cell 6 — DABL-C Loss + EDL + CLAAM + CLAC integration
class DABLCLoss(nn.Module):
    """DABL-C loss + Novel contributions:
    - Original: asymmetric BCE + Dice + Boundary sharpening + Confidence supervision
    - NEW: Constrained learnable weights (ratio-based, prevents collapse)
    - NEW: Gap-aware boundary loss (penalizes inter-building merging)
    - NEW: EDL Bayes risk + KL (Contribution 1)
    - NEW: CLAAM consistency + diversity regularization (Contribution 2 / MLAA)
    - NEW: CLAC spatial consistency regularization (Idea 2)
    """
    def __init__(self, w_in_init=1.0, w_out_init=1.0, dice_weight=1.0,
                 boundary_weight=0.5, conf_loss_weight=0.3):
        super().__init__()
        # Constrained parameterization: learn the LOG-RATIO of w_in/w_out
        # Total sum is fixed at (w_in_init + w_out_init), only the balance is learned.
        # This prevents both weights from collapsing to zero.
        self._total_w = w_in_init + w_out_init  # fixed constant
        # logit_ratio: sigmoid(logit) = w_in / (w_in + w_out)
        init_ratio = w_in_init / (w_in_init + w_out_init)
        init_logit = math.log(init_ratio / (1.0 - init_ratio + 1e-8) + 1e-8)
        self._logit_ratio = nn.Parameter(torch.tensor(float(init_logit)))
        self.dice_weight = dice_weight
        self.boundary_weight = boundary_weight
        self.conf_loss_weight = conf_loss_weight

    @property
    def w_in(self):
        """Building weight: derived from learned ratio, sum-constrained."""
        ratio = torch.sigmoid(self._logit_ratio)
        return self._total_w * ratio

    @property
    def w_out(self):
        """Background weight: derived from learned ratio, sum-constrained."""
        ratio = torch.sigmoid(self._logit_ratio)
        return self._total_w * (1.0 - ratio)

    def _boundary(self, x):
        """Gradient-based boundary extraction."""
        gx = x[:, :, :, 1:] - x[:, :, :, :-1]
        gy = x[:, :, 1:, :] - x[:, :, :-1, :]
        gx = F.pad(gx, (0, 1, 0, 0))
        gy = F.pad(gy, (0, 0, 0, 1))
        return (gx.abs() + gy.abs()).clamp(0, 1)

    def _gap_weight_map(self, target):
        """Create weight map emphasizing inter-building gaps.
        Uses GPU-based max-pool dilation to find gap pixels between
        adjacent buildings — these are the regions where merging errors
        occur and boundary accuracy is most critical.
        Returns: weight map [B, 1, H, W] with higher values at gaps."""
        # Dilate GT buildings on GPU using max pooling (equivalent to morphological dilation)
        dilated = F.max_pool2d(target, kernel_size=11, stride=1, padding=5)
        # Gap pixels: near buildings but not buildings themselves
        gap = (dilated - target).clamp(0, 1)
        # Building boundary pixels: buildings minus eroded buildings
        eroded = -F.max_pool2d(-target, kernel_size=5, stride=1, padding=2)  # min-pool = erosion
        boundary = (target - eroded).clamp(0, 1)
        # Weight map: base=1, gap regions=5x, building boundaries=3x
        weight = 1.0 + 4.0 * gap + 2.0 * boundary
        return weight

    def _claam_diversity_loss(self, layer_logits):
        """Penalize excessive agreement among CLAAM per-layer predictions.
        Encourages each ViT layer head to capture different building aspects.
        Uses pairwise cosine similarity on downsampled predictions."""
        if len(layer_logits) < 2:
            return torch.tensor(0.0, device=layer_logits[0].device)
        flat = [F.adaptive_avg_pool2d(torch.sigmoid(l), (16, 16)).reshape(l.shape[0], -1)
                for l in layer_logits]
        sim_sum = 0.0
        count = 0
        for i in range(len(flat)):
            for j in range(i + 1, len(flat)):
                sim_sum = sim_sum + F.cosine_similarity(flat[i], flat[j], dim=-1).mean()
                count += 1
        return sim_sum / max(count, 1)

    def forward(self, logits, target, conf=None, model_output=None,
                epoch=0, total_epochs=45, is_pseudo=False):
        probs = torch.sigmoid(logits)
        eps = 1e-6
        w_in = self.w_in     # derived from constrained ratio
        w_out = self.w_out

        # Dynamic pos_weight: small buildings get drowned by background
        pos_pixels = target.sum() + eps
        neg_pixels = (1 - target).sum() + eps
        pos_weight = (neg_pixels / pos_pixels).clamp(max=5.0)

        # Asymmetric BCE with area-based reweighting
        bce = -(w_in * pos_weight * target * torch.log(probs + eps) +
                w_out * (1 - target) * torch.log(1 - probs + eps))
        if conf is not None:
            bce = bce * conf
        bce_loss = bce.mean()

        # Dice loss
        inter = (probs * target).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice_loss = (1.0 - (2.0 * inter + eps) / (union + eps)).mean()

        # Gap-aware boundary loss: steep sigmoid → near-hard edges
        # Weight boundary errors MORE heavily in inter-building gap regions
        # This directly penalizes the under-segmentation / merging problem
        sharp_probs = torch.sigmoid(logits * 10.0)
        pred_bd = self._boundary(sharp_probs)
        gt_bd = self._boundary(target)
        gap_weights = self._gap_weight_map(target)
        bd_loss = (gap_weights * (pred_bd - gt_bd) ** 2).mean()

        total = bce_loss + self.dice_weight * dice_loss + self.boundary_weight * bd_loss

        # Scale factor for novel loss terms during pseudo-label training
        novel_scale = 0.3 if is_pseudo else 1.0

        # ── CONTRIBUTION 1: EDL Loss (Bayes risk + annealed KL) ──
        if model_output is not None and model_output.get("edl") is not None:
            alpha = model_output["edl"]["alpha"]
            l_edl = edl_loss(alpha, target, epoch, total_epochs,
                             annealing_epochs=cfg.edl_annealing_epochs)
            total = total + cfg.edl_loss_weight * novel_scale * l_edl

        # ── CONTRIBUTION 2: CLAAM Consistency + Diversity Loss ──
        if model_output is not None and model_output.get("claam") is not None:
            claam_out = model_output["claam"]
            l_claam = claam_consistency_loss(claam_out["layer_logits"], target)
            total = total + cfg.claam_loss_weight * novel_scale * l_claam
            l_div = self._claam_diversity_loss(claam_out["layer_logits"])
            total = total + cfg.claam_diversity_weight * l_div

        # ── IDEA 2: CLAC-Loss (cross-layer spatial consistency regularization) ──
        if model_output is not None and model_output.get("multi_feats") is not None:
            l_clac = clac_loss(model_output["multi_feats"])
            total = total + cfg.clac_loss_weight * l_clac

        # ── IDEA 5: SDR Loss (signed distance field regression) ──
        if model_output is not None and model_output.get("sdf_pred") is not None:
            sdf_pred = model_output["sdf_pred"]
            sdf_target = compute_sdf_target(target, max_dist=cfg.sdr_max_dist)
            l_sdr = F.smooth_l1_loss(sdf_pred, sdf_target)
            total = total + cfg.sdr_loss_weight * novel_scale * l_sdr

        return total

In [ ]:
# Cell 7 — Progressive threshold + Multi-Uncertainty Gate (7 gates) + TVR

def progressive_threshold(epoch: int, total_epochs: int, start: float, end: float) -> float:
    if total_epochs <= 1:
        return end
    t = epoch / (total_epochs - 1)
    return start + t * (end - start)


def _geometry_gate(probs: torch.Tensor, thr: float = 0.5,
                   compact_min: float = 0.15, rect_min: float = 0.4) -> torch.Tensor:
    """Per-pixel geometry gate: reject connected components that are
    not building-like (too irregular or too elongated)."""
    import cv2
    B = probs.shape[0]
    gate = torch.ones_like(probs, dtype=torch.bool)
    for b in range(B):
        mask_np = (probs[b, 0].detach().cpu().numpy() > thr).astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        reject_mask = np.zeros_like(mask_np, dtype=np.uint8)
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area < 10:
                continue
            peri = cv2.arcLength(cnt, True)
            compactness = (4 * math.pi * area) / (peri * peri + 1e-6)
            x, y, w, h = cv2.boundingRect(cnt)
            rectangularity = area / (w * h + 1e-6)
            if compactness < compact_min or rectangularity < rect_min:
                cv2.drawContours(reject_mask, [cnt], -1, 255, thickness=cv2.FILLED)
        if reject_mask.any():
            reject_t = torch.from_numpy(reject_mask > 0).to(probs.device)
            gate[b, 0] = gate[b, 0] & (~reject_t)
    return gate


def _erode_pseudo_labels(pseudo: torch.Tensor, erode_px: int) -> torch.Tensor:
    """Erode pseudo-label boundaries by erode_px pixels using GPU min-pooling.
    This PREVENTS the merging echo-chamber: by shrinking each predicted building
    a few pixels inward, we ensure the model never reinforces the 'bleed' across
    narrow inter-building gaps from its own noisy predictions.
    Only the confident interior pixels survive as pseudo-labels."""
    if erode_px <= 0:
        return pseudo
    k = 2 * erode_px + 1
    # Min-pooling = erosion for binary masks: shrinks foreground by erode_px
    eroded = -F.max_pool2d(-pseudo, kernel_size=k, stride=1, padding=erode_px)
    return eroded


# ═══════════════════════════════════════════════════════════════════
#  IDEA 3: Temporal Voting across Rotations (TVR)
# ═══════════════════════════════════════════════════════════════════

class TemporalVotingBuffer:
    """Maintains a rolling buffer of pseudo-label predictions across
    label rotation cycles. Pixels must be consistently predicted as
    building across tvr_window cycles to be accepted."""
    def __init__(self, window: int = 5, accept_ratio: float = 0.6):
        self.window = window
        self.accept_ratio = accept_ratio
        self.buffer = {}

    def update(self, image_ids: list, predictions: torch.Tensor):
        from collections import deque
        for i, img_id in enumerate(image_ids):
            if img_id not in self.buffer:
                self.buffer[img_id] = deque(maxlen=self.window)
            self.buffer[img_id].append(predictions[i:i+1].detach().cpu())

    def get_stable_mask(self, image_ids: list, device: torch.device) -> torch.Tensor:
        masks = []
        for img_id in image_ids:
            if img_id not in self.buffer or len(self.buffer[img_id]) < 2:
                masks.append(torch.ones(1, 1, 1, 1))
            else:
                stacked = torch.cat(list(self.buffer[img_id]), dim=0)
                vote_frac = stacked.mean(dim=0, keepdim=True)
                stable = (vote_frac >= self.accept_ratio).float()
                masks.append(stable)
        result = []
        for m in masks:
            result.append(m.to(device))
        return result


tvr_buffer = TemporalVotingBuffer(
    window=cfg.tvr_window,
    accept_ratio=cfg.tvr_accept_ratio
)


# ═══════════════════════════════════════════════════════════════════
#  Legacy triple_gate_accept (kept for backward compat / ablation)
# ═══════════════════════════════════════════════════════════════════

def triple_gate_accept(probs: torch.Tensor, conf: torch.Tensor,
                       edge: torch.Tensor, thr: float) -> torch.Tensor:
    gate_conf = conf >= thr
    edge_strength = torch.sigmoid(edge)
    gate_edge = edge_strength <= 0.6
    gate_geom = _geometry_gate(probs, thr=0.5)
    gate = gate_conf & gate_edge & gate_geom
    b = probs.shape[0]
    pseudo = torch.zeros_like(probs)
    for i in range(b):
        p = probs[i]
        p_mean = p.mean()
        p_std = p.std()
        adaptive_thr = max(p_mean + 0.5 * p_std, 0.2)
        pseudo[i] = (p >= adaptive_thr).float()
    pseudo = pseudo * gate.float()
    return pseudo


# ═══════════════════════════════════════════════════════════════════
#  NEW: Multi-Uncertainty Gate (7 gates — all contributions combined)
# ═══════════════════════════════════════════════════════════════════

def multi_uncertainty_gate(probs: torch.Tensor, model_output: dict,
                           thr: float, epoch: int, total_epochs: int,
                           image_ids: list = None) -> torch.Tensor:
    """7-gate pseudo-label acceptance + boundary erosion anti-merging.
    Triple-source uncertainty:
    1. Output-level: EDL epistemic uncertainty
    2. Encoder-level: CLAAM cross-layer attention agreement (MLAA)
    3. Training-dynamics-level: TVR temporal voting across rotations
    Plus domain-specific priors: edge, geometry, GOP, CAFCG.
    Plus boundary erosion to break merging echo chamber.
    """
    conf = model_output.get("conf")
    edge = model_output.get("edge")
    edl_out = model_output.get("edl")
    claam_out = model_output.get("claam")

    B = probs.shape[0]

    # ── Gate 1: EDL uncertainty (widened thresholds for building interiors) ──
    edl_thr = progressive_threshold(epoch, total_epochs, 0.35, 0.55)
    gate_conf = edl_out["uncertainty"] < edl_thr

    # ── Gate 2: Edge guidance (tightened: reject pixels near predicted edges) ──
    edge_strength = torch.sigmoid(edge)
    gate_edge = edge_strength <= 0.45  # Tightened from 0.6: reject more boundary pixels

    # ── Gate 3: Geometry (compactness + rectangularity) ──
    gate_geom = _geometry_gate(probs, thr=0.5)

    # ── Gate 4: CLAAM / MLAA agreement (Contribution 2) ──
    claam_thr = progressive_threshold(epoch, total_epochs,
                                       cfg.claam_thr_start, cfg.claam_thr_end)
    gate_claam = claam_out["agreement"] >= claam_thr

    # ── Gate 5: TVR temporal stability (Idea 3) ──
    if image_ids is not None:
        tvr_masks = tvr_buffer.get_stable_mask(image_ids, probs.device)
        gate_tvr_list = []
        for i in range(B):
            m = tvr_masks[i] if i < len(tvr_masks) else torch.ones(1, 1, 1, 1, device=probs.device)
            if m.shape[-2:] != probs.shape[-2:]:
                m = torch.ones(1, 1, probs.shape[2], probs.shape[3], device=probs.device)
            gate_tvr_list.append(m > 0.5)
        gate_tvr = torch.cat(gate_tvr_list, dim=0)
    else:
        gate_tvr = torch.ones_like(gate_conf, dtype=torch.bool)

    # ── Gate 6: GOP gradient orientation prior (Idea 4) ──
    gate_gop = gradient_orientation_gate(probs, thr=0.5,
                                          rectilinear_thr=cfg.gop_rectilinear_thr)

    # ── Gate 7: CAFCG frequency consistency (Idea 5) ──
    gate_cafcg = cafcg_gate(probs, disagreement_thr=cfg.cafcg_disagreement_thr)

    # ── Combine all gates ──
    gate = gate_conf & gate_edge & gate_geom & gate_claam & gate_tvr & gate_gop & gate_cafcg

    # ── Adaptive per-image thresholding for pseudo-label generation ──
    pseudo = torch.zeros_like(probs)
    for i in range(B):
        p = probs[i]
        p_mean = p.mean()
        p_std = p.std()
        adaptive_thr = max(p_mean + 0.5 * p_std, 0.2)
        pseudo[i] = (p >= adaptive_thr).float()

    # Apply all gates
    pseudo = pseudo * gate.float()

    # ── ANTI-MERGING: Erode pseudo-label boundaries ──
    # Shrinks predicted buildings inward by cfg.pseudo_edge_erode_px pixels.
    # The narrow inter-building gaps (where merging happens) are removed
    # from training targets, breaking the echo-chamber feedback loop.
    pseudo = _erode_pseudo_labels(pseudo, cfg.pseudo_edge_erode_px)

    # ── TVR: update buffer with this cycle's pseudo-labels ──
    if image_ids is not None:
        tvr_buffer.update(image_ids, pseudo)

    return pseudo

In [ ]:
# Cell 8
from scipy.ndimage import binary_dilation, binary_erosion, distance_transform_edt

def boundary_iou(pred: torch.Tensor, target: torch.Tensor,
                 thr: float = 0.5, dilation_px: int = 3, eps: float = 1e-6) -> torch.Tensor:
    """Boundary-IoU following Cheng et al. 2021.
    Uses morphological dilation to extract boundary regions at fixed pixel tolerance,
    then computes IoU only within those boundary strips."""
    pred_bin = (pred >= thr).float()
    b = pred_bin.shape[0]
    struct = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1))
    biou_vals = []
    for i in range(b):
        p = pred_bin[i, 0].cpu().numpy()  # [H, W]
        t = target[i, 0].cpu().numpy()
        # Boundary = dilated XOR original (the boundary strip)
        p_dilated = binary_dilation(p, structure=struct).astype(np.float32)
        t_dilated = binary_dilation(t, structure=struct).astype(np.float32)
        p_eroded = binary_erosion(p, structure=struct).astype(np.float32)
        t_eroded = binary_erosion(t, structure=struct).astype(np.float32)
        p_boundary = np.clip((p_dilated - p) + (p - p_eroded), 0, 1)
        t_boundary = np.clip((t_dilated - t) + (t - t_eroded), 0, 1)
        # IoU within boundary regions
        inter = (p_boundary * t_boundary).sum()
        union = p_boundary.sum() + t_boundary.sum() - inter
        biou_vals.append((inter + eps) / (union + eps))
    return torch.tensor(np.mean(biou_vals), device=pred.device)


def standard_iou(pred: torch.Tensor, target: torch.Tensor,
                 thr: float = 0.5, eps: float = 1e-6) -> torch.Tensor:
    """Standard pixel-level IoU (Jaccard index)."""
    pred_bin = (pred >= thr).float()
    inter = (pred_bin * target).sum(dim=(1, 2, 3))
    union = pred_bin.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) - inter
    return ((inter + eps) / (union + eps)).mean()


def hausdorff_distance(pred: torch.Tensor, target: torch.Tensor,
                       thr: float = 0.5, percentile: float = 95.0) -> torch.Tensor:
    """Hausdorff distance via boundary distance transforms (robust: uses percentile).
    Uses boundary pixels (not filled regions) for standard HD95 computation.
    Properly handles empty pred/GT cases."""
    pred_bin = (pred >= thr).float()
    b = pred_bin.shape[0]
    hd_vals = []
    for i in range(b):
        p = pred_bin[i, 0].cpu().numpy().astype(bool)
        t = target[i, 0].cpu().numpy().astype(bool)
        # Both empty → perfect agreement → HD = 0
        if not p.any() and not t.any():
            hd_vals.append(0.0)
            continue
        # One empty, other not → max possible distance as penalty (capped)
        if not p.any() or not t.any():
            hd_vals.append(float(np.sqrt(p.shape[0]**2 + p.shape[1]**2)))
            continue
        # Extract boundaries for standard HD95
        p_boundary = p ^ binary_erosion(p)
        t_boundary = t ^ binary_erosion(t)
        if not p_boundary.any():
            p_boundary = p  # single-pixel components
        if not t_boundary.any():
            t_boundary = t
        dt_pred = distance_transform_edt(~p_boundary)
        dt_tgt = distance_transform_edt(~t_boundary)
        d_t2p = dt_pred[t_boundary]   # distances from GT boundary to nearest pred boundary
        d_p2t = dt_tgt[p_boundary]    # distances from pred boundary to nearest GT boundary
        hd = max(np.percentile(d_t2p, percentile), np.percentile(d_p2t, percentile))
        hd_vals.append(float(hd))
    return torch.tensor(np.mean(hd_vals), device=pred.device)


def assd_distance(pred: torch.Tensor, target: torch.Tensor,
                  thr: float = 0.5) -> torch.Tensor:
    """Average Symmetric Surface Distance via distance transforms.
    ASSD = mean of all surface-to-surface distances in both directions.
    Properly handles empty pred/GT cases."""
    pred_bin = (pred >= thr).float()
    b = pred_bin.shape[0]
    assd_vals = []
    for i in range(b):
        p = pred_bin[i, 0].cpu().numpy().astype(bool)
        t = target[i, 0].cpu().numpy().astype(bool)
        # Both empty → perfect agreement → ASSD = 0
        if not p.any() and not t.any():
            assd_vals.append(0.0)
            continue
        # One empty, other not → max penalty
        if not p.any() or not t.any():
            assd_vals.append(float(np.sqrt(p.shape[0]**2 + p.shape[1]**2)))
            continue
        p_boundary = p ^ binary_erosion(p)
        t_boundary = t ^ binary_erosion(t)
        if not p_boundary.any():
            p_boundary = p
        if not t_boundary.any():
            t_boundary = t
        dt_pred = distance_transform_edt(~p_boundary)
        dt_tgt = distance_transform_edt(~t_boundary)
        d_t2p = dt_pred[t_boundary].mean() if t_boundary.any() else 0.0
        d_p2t = dt_tgt[p_boundary].mean() if p_boundary.any() else 0.0
        assd_vals.append(float((d_t2p + d_p2t) / 2.0))
    return torch.tensor(np.mean(assd_vals), device=pred.device)


def ece_score(probs: torch.Tensor, target: torch.Tensor, n_bins: int = 15) -> torch.Tensor:
    conf = probs.view(-1)
    t = target.view(-1)
    bins = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    ece = torch.zeros(1, device=probs.device)
    for i in range(n_bins):
        in_bin = (conf >= bins[i]) & (conf < bins[i + 1])
        if in_bin.any():
            acc = t[in_bin].mean()
            avg_conf = conf[in_bin].mean()
            ece += (in_bin.float().mean()) * (acc - avg_conf).abs()
    return ece


def reliability_bins(probs: torch.Tensor, target: torch.Tensor, n_bins: int = 15):
    conf = probs.view(-1)
    t = target.view(-1)
    bins = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    bin_acc, bin_conf, bin_frac = [], [], []
    for i in range(n_bins):
        in_bin = (conf >= bins[i]) & (conf < bins[i + 1])
        if in_bin.any():
            bin_acc.append(t[in_bin].mean().item())
            bin_conf.append(conf[in_bin].mean().item())
            bin_frac.append(in_bin.float().mean().item())
        else:
            bin_acc.append(0.0)
            bin_conf.append(0.0)
            bin_frac.append(0.0)
    return bin_acc, bin_conf, bin_frac

In [ ]:
# Cell 9
class RandomSegDataset(Dataset):
    def __init__(self, length: int = 10, img_size: int = 512):
        self.length = length
        self.img_size = img_size

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        img = torch.rand(3, self.img_size, self.img_size)
        mask = (torch.rand(1, self.img_size, self.img_size) > 0.5).float()
        return img, mask


def load_images(folder: str):
    exts = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.bmp"]
    paths = []
    for ext in exts:
        paths += glob.glob(os.path.join(folder, ext))
    return sorted(paths)


DATA_ROOT = "/kaggle/input/datasets/sengulgs/whu-building-dataset/WHU"
TRAIN_IMG = os.path.join(DATA_ROOT, "train", "Image")
TRAIN_MASK = os.path.join(DATA_ROOT, "train", "Mask")
VAL_IMG = os.path.join(DATA_ROOT, "val", "Image")
VAL_MASK = os.path.join(DATA_ROOT, "val", "Mask")
TEST_IMG = os.path.join(DATA_ROOT, "test", "Image")
TEST_MASK = os.path.join(DATA_ROOT, "test", "Mask")


class WHUDataset(Dataset):
    """WHU Building dataset with optional joint augmentation."""
    def __init__(self, image_paths, mask_paths=None, img_size=512, augment=False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def _augment(self, img, mask=None):
        """Joint spatial + photometric augmentation for image (and mask)."""
        if random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            if mask is not None:
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() > 0.5:
            img = img.transpose(Image.FLIP_TOP_BOTTOM)
            if mask is not None:
                mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
        k = random.randint(0, 3)
        if k == 1:
            img = img.transpose(Image.ROTATE_90)
            if mask is not None: mask = mask.transpose(Image.ROTATE_90)
        elif k == 2:
            img = img.transpose(Image.ROTATE_180)
            if mask is not None: mask = mask.transpose(Image.ROTATE_180)
        elif k == 3:
            img = img.transpose(Image.ROTATE_270)
            if mask is not None: mask = mask.transpose(Image.ROTATE_270)

        img_np = np.array(img).astype(np.float32)
        img_np = img_np * random.uniform(0.8, 1.2)
        mean_val = img_np.mean()
        img_np = (img_np - mean_val) * random.uniform(0.8, 1.2) + mean_val
        gray = img_np.mean(axis=2, keepdims=True)
        sat = random.uniform(0.7, 1.3)
        img_np = img_np * sat + gray * (1 - sat)
        img_np = np.clip(img_np, 0, 255).astype(np.uint8)
        img = Image.fromarray(img_np)
        return img, mask

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = img.resize((self.img_size, self.img_size), resample=Image.BILINEAR)

        mask = None
        if self.mask_paths:
            mask = Image.open(self.mask_paths[idx]).convert("L")
            mask = mask.resize((self.img_size, self.img_size), resample=Image.NEAREST)

        if self.augment:
            img, mask = self._augment(img, mask)

        img = torch.tensor(np.array(img) / 255.0).permute(2, 0, 1).float()

        if mask is not None:
            mask = (np.array(mask) > 127).astype(np.float32)
            mask = torch.tensor(mask).unsqueeze(0)
            return img, mask

        return img


# ── Store ALL train paths globally for label rotation ──
ALL_TRAIN_IMG = []
ALL_TRAIN_MASK = []

def rotate_train_loaders():
    """Re-sample a RANDOM 10% as labeled each call.
    Over many epochs, the model eventually sees every image with its label."""
    n = len(ALL_TRAIN_IMG)
    n_labeled = max(1, int(n * cfg.label_frac))
    indices = list(range(n))
    random.shuffle(indices)
    lab_idx = indices[:n_labeled]
    unl_idx = indices[n_labeled:]

    labeled_img  = [ALL_TRAIN_IMG[i] for i in lab_idx]
    labeled_mask = [ALL_TRAIN_MASK[i] for i in lab_idx]
    unlabeled_img = [ALL_TRAIN_IMG[i] for i in unl_idx]

    lab_ds = WHUDataset(labeled_img, labeled_mask, img_size=cfg.img_size, augment=True)
    unl_ds = WHUDataset(unlabeled_img, None, img_size=cfg.img_size, augment=True)
    lab_loader = DataLoader(lab_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    unl_loader = DataLoader(unl_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    return lab_loader, unl_loader


def build_dataloaders():
    global ALL_TRAIN_IMG, ALL_TRAIN_MASK
    if os.path.isdir(DATA_ROOT):
        train_img = load_images(TRAIN_IMG)
        train_mask = load_images(TRAIN_MASK)
        val_img = load_images(VAL_IMG)
        val_mask = load_images(VAL_MASK)

        n_train = min(len(train_img), len(train_mask))
        ALL_TRAIN_IMG = train_img[:n_train]
        ALL_TRAIN_MASK = train_mask[:n_train]

        n_val = min(len(val_img), len(val_mask))
        val_img = val_img[:n_val]
        val_mask = val_mask[:n_val]

        val_ds = WHUDataset(val_img, val_mask, img_size=cfg.img_size, augment=False)
    else:
        print("WHU paths not found, using random dataset.")
        val_ds = RandomSegDataset(length=2, img_size=cfg.img_size)

    # Initial random split
    labeled_loader, unlabeled_loader = rotate_train_loaders() if ALL_TRAIN_IMG else (
        DataLoader(RandomSegDataset(length=4, img_size=cfg.img_size), batch_size=cfg.batch_size, shuffle=True),
        DataLoader(RandomSegDataset(length=2, img_size=cfg.img_size), batch_size=cfg.batch_size, shuffle=True),
    )
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
    print(f"Total train images: {len(ALL_TRAIN_IMG)}, "
          f"labeled per epoch: {max(1, int(len(ALL_TRAIN_IMG) * cfg.label_frac))}")
    return labeled_loader, unlabeled_loader, val_loader


labeled_loader, unlabeled_loader, val_loader = build_dataloaders()


In [ ]:
# Cell 10 — Training loop (updated for EDL/CLAAM/CLAC/TVR/GOP/CAFCG)
import copy
import cv2
from scipy import ndimage as ndi
from scipy.ndimage import uniform_filter, label as cc_label
import hashlib

# ──────────────────────── Post-processing ────────────────────────

def morpho_clean(pred_np, ksize=3):
    """Morphological close only (fill holes). No opening — it destroys small buildings."""
    struct = np.ones((ksize, ksize))
    pred_np = ndi.binary_closing(pred_np, structure=struct).astype(np.float32)
    return pred_np


def watershed_separate(binary_np, min_distance=8, min_area=30):
    """Watershed-based building separation to split merged buildings.
    Uses distance transform + local maxima as markers."""
    from scipy.ndimage import label as nd_label, distance_transform_edt
    if binary_np.sum() < min_area:
        return binary_np
    dist = distance_transform_edt(binary_np)
    from scipy.ndimage import maximum_filter
    local_max = (dist == maximum_filter(dist, size=min_distance * 2 + 1))
    local_max = local_max & (dist > 2)
    markers, num_markers = nd_label(local_max)
    if num_markers < 2:
        return binary_np
    from skimage.segmentation import watershed
    labels = watershed(-dist, markers, mask=binary_np.astype(bool))
    result = np.zeros_like(binary_np, dtype=np.float32)
    for lbl in range(1, num_markers + 1):
        component = (labels == lbl).astype(np.float32)
        if component.sum() >= min_area:
            result = np.maximum(result, component)
    return result


def polygon_refine(binary_np, epsilon_frac=0.005, min_area=20):
    """Simplify each connected component's contour to a polygon."""
    mask = (binary_np * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result = np.zeros_like(binary_np, dtype=np.float32)
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue
        peri = cv2.arcLength(cnt, True)
        if area < 200:
            epsilon = 0.002 * peri
        elif area < 800:
            epsilon = 0.004 * peri
        else:
            epsilon = epsilon_frac * peri
        approx = cv2.approxPolyDP(cnt, epsilon, True)
        cv2.drawContours(result, [approx], -1, 1.0, thickness=cv2.FILLED)
    return result


def refine_prediction(img_np, prob_np, thr=0.5, min_area=20):
    binary = (prob_np > thr).astype(np.float32)
    binary = morpho_clean(binary, ksize=3)
    binary = watershed_separate(binary, min_distance=8, min_area=min_area)
    sharp = polygon_refine(binary, epsilon_frac=0.005, min_area=min_area)
    return prob_np, sharp


def guided_filter(guide, src, radius=8, eps=0.01):
    g = guide.mean(axis=2).astype(np.float64) if guide.ndim == 3 else guide.astype(np.float64)
    s = src.astype(np.float64)
    sz = 2 * radius + 1
    mean_g  = uniform_filter(g, size=sz)
    mean_s  = uniform_filter(s, size=sz)
    mean_gs = uniform_filter(g * s, size=sz)
    mean_gg = uniform_filter(g * g, size=sz)
    a = (mean_gs - mean_g * mean_s) / (mean_gg - mean_g * mean_g + eps)
    b = mean_s - a * mean_g
    mean_a = uniform_filter(a, size=sz)
    mean_b = uniform_filter(b, size=sz)
    return np.clip(mean_a * g + mean_b, 0, 1).astype(np.float32)


def _make_image_ids(imgs: torch.Tensor) -> list:
    ids = []
    for i in range(imgs.shape[0]):
        small = F.interpolate(imgs[i:i+1], size=(32, 32), mode='bilinear', align_corners=False)
        h = hashlib.md5(small.cpu().numpy().tobytes()).hexdigest()[:12]
        ids.append(h)
    return ids


# ──────────────────────── Training ────────────────────────

def train_one_epoch(model, labeled_loader, optimizer, loss_fn, epoch,
                    total_epochs, use_pseudo=False, unlabeled_loader=None):
    model.train()
    total_loss, steps = 0.0, 0
    thr = progressive_threshold(epoch, total_epochs, cfg.thr_start, cfg.thr_end)

    # Pseudo-label loss ramp-up: start gentle, increase over Stage 2 epochs
    pseudo_ramp = min(1.0, cfg.pseudo_ramp_start + (1.0 - cfg.pseudo_ramp_start) * epoch / max(total_epochs - 1, 1))

    # ── Pseudo-label stage (unlabeled data) ──
    if use_pseudo and unlabeled_loader is not None:
        pbar = tqdm(unlabeled_loader, desc=f"pseudo e{epoch} ramp={pseudo_ramp:.2f}", leave=True, dynamic_ncols=True)
        for batch in pbar:
            imgs = batch[0] if isinstance(batch, (list, tuple)) else batch
            imgs = imgs.to(device)
            out = model(imgs)
            logits = out["logits"]
            conf = out.get("conf")
            probs = torch.sigmoid(logits)

            # ── 7-gate + erosion pseudo-label acceptance ──
            image_ids = _make_image_ids(imgs)
            target = multi_uncertainty_gate(
                probs, out, thr, epoch, total_epochs, image_ids=image_ids)

            if target.sum() < 1.0:
                continue

            # ── Full loss with is_pseudo=True (scales down EDL/CLAAM) ──
            loss = loss_fn(logits, target, conf,
                           model_output=out, epoch=epoch,
                           total_epochs=total_epochs, is_pseudo=True)

            # Apply pseudo-label ramp-up weight
            loss = loss * pseudo_ramp

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(loss_fn.parameters()), max_norm=2.0)
            optimizer.step()
            total_loss += loss.item(); steps += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

    # ── Supervised stage (labeled data) ──
    pbar = tqdm(labeled_loader, desc=f"train e{epoch}", leave=True, dynamic_ncols=True)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        out = model(imgs)
        conf = out.get("conf")

        loss = loss_fn(out["logits"], masks, conf,
                       model_output=out, epoch=epoch,
                       total_epochs=total_epochs, is_pseudo=False)

        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.parameters()) + list(loss_fn.parameters()), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item(); steps += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(1, steps)


def validate(model, loader):
    model.eval()
    biou_all, iou_all, ece_all = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            probs = torch.sigmoid(model(imgs)["logits"])
            biou_all.append(boundary_iou(probs, masks).item())
            iou_all.append(standard_iou(probs, masks).item())
            ece_all.append(ece_score(probs, masks).item())
    return float(np.mean(biou_all)), float(np.mean(iou_all)), float(np.mean(ece_all))


def preview_prediction(model, loader, samples=2):
    """Show RANDOM samples with polygon-refined predictions."""
    import matplotlib.pyplot as plt
    model.eval()
    dataset = loader.dataset
    n = len(dataset)
    indices = random.sample(range(n), min(samples, n))
    imgs_list, masks_list = [], []
    for idx in indices:
        item = dataset[idx]
        imgs_list.append(item[0])
        masks_list.append(item[1])
    imgs = torch.stack(imgs_list).to(device)
    masks = torch.stack(masks_list).to(device)

    with torch.no_grad():
        probs = torch.sigmoid(model(imgs)["logits"])
    print("prob stats:", probs.min().item(), probs.mean().item(), probs.max().item())
    for i in range(imgs.shape[0]):
        img_np  = imgs[i].permute(1, 2, 0).cpu().numpy()
        gt_np   = masks[i, 0].cpu().numpy()
        prob_np = probs[i, 0].cpu().numpy()
        raw_pred = (prob_np > cfg.pred_thr).astype(np.float32)
        _, polygon = refine_prediction(img_np, prob_np, thr=cfg.pred_thr)

        fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
        for ax, title, data, cmap in zip(
            axes,
            ["Image", "GT", "Prob", f"Raw@{cfg.pred_thr}", "Polygon"],
            [img_np, gt_np, prob_np, raw_pred, polygon],
            [None, "gray", "magma", "gray", "gray"]
        ):
            ax.set_title(title, fontsize=11); ax.imshow(data, cmap=cmap); ax.axis("off")
        plt.tight_layout(); plt.show()


def run_stage(model, optimizer, scheduler, loss_fn, val_loader,
              total_epochs, stage_name, best_biou, best_state,
              use_pseudo=False):
    """Each epoch: rotate which 10% is labeled → model sees NEW images with labels."""
    no_improve = 0
    for epoch in range(total_epochs):
        labeled_loader, unlabeled_loader = rotate_train_loaders()
        loss = train_one_epoch(model, labeled_loader, optimizer, loss_fn,
                               epoch, total_epochs, use_pseudo=use_pseudo,
                               unlabeled_loader=unlabeled_loader)
        biou, iou, ece = validate(model, val_loader)
        # Log DABL-C learnable weights each epoch (uses property accessors)
        w_in_val = loss_fn.w_in.item() if hasattr(loss_fn, 'w_in') else None
        w_out_val = loss_fn.w_out.item() if hasattr(loss_fn, 'w_out') else None
        w_str = f" w_in={w_in_val:.3f} w_out={w_out_val:.3f}" if w_in_val is not None else ""
        print(f"  e{epoch} loss={loss:.4f} biou={biou:.4f} iou={iou:.4f} ece={ece:.4f}{w_str}")
        if w_in_val is not None:
            weight_history["epoch"].append(global_epoch_counter[0])
            weight_history["stage"].append(stage_name)
            weight_history["w_in"].append(w_in_val)
            weight_history["w_out"].append(w_out_val)
            global_epoch_counter[0] += 1
        if scheduler is not None:
            scheduler.step()
        if biou > best_biou:
            best_biou = biou
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, cfg.save_path)
            print(f"  ** saved (biou={best_biou:.4f} iou={iou:.4f})")
            no_improve = 0
        else:
            no_improve += 1
        if cfg.preview_every > 0 and (epoch + 1) % cfg.preview_every == 0:
            preview_prediction(model, val_loader, samples=cfg.preview_samples)
        if no_improve >= cfg.patience:
            print(f"  early stopping at epoch {epoch}")
            break
    return best_biou, best_state


# ==================== Build model (with novel contributions) ====================
weight_history = {"epoch": [], "stage": [], "w_in": [], "w_out": []}
global_epoch_counter = [0]

model = DABLCNet().to(device)

loss_fn = DABLCLoss(
    w_in_init=cfg.loss_w_in, w_out_init=cfg.loss_w_out,
    dice_weight=cfg.loss_dice_w,
    boundary_weight=cfg.boundary_weight,
    conf_loss_weight=cfg.conf_loss_weight,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
loss_params_count = sum(p.numel() for p in loss_fn.parameters())
print(f"Model params: {total_params:,} total, {trainable_params:,} trainable")
print(f"Loss learnable params: {loss_params_count} (logit_ratio for DABL-C w_in/w_out)")
print(f"Initial DABL-C: w_in={loss_fn.w_in.item():.3f}, w_out={loss_fn.w_out.item():.3f}")
print(f"Novel modules: EDL head = {model.edl_head is not None}, "
      f"CLAAM = {model.claam is not None}")

# ===== Warmup: freeze ViT backbone, let decoder/heads learn first =====
print(f"\nWarmup: backbone frozen for {cfg.warmup_freeze} epochs")
for p in model.encoder.parameters():
    p.requires_grad = False
head_only = [p for p in model.parameters() if p.requires_grad]
head_only = head_only + list(loss_fn.parameters())
warmup_opt = torch.optim.AdamW(head_only, lr=cfg.lr_head)
best_biou, best_state = 0.0, None
for we in range(cfg.warmup_freeze):
    lab_ldr, _ = rotate_train_loaders()
    wloss = train_one_epoch(model, lab_ldr, warmup_opt, loss_fn, we, cfg.warmup_freeze)
    biou, iou, ece = validate(model, val_loader)
    print(f"  warmup e{we} loss={wloss:.4f} biou={biou:.4f} iou={iou:.4f}")
    if biou > best_biou:
        best_biou = biou
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, cfg.save_path)
for p in model.encoder.parameters():
    p.requires_grad = True
print("Backbone unfrozen -> full training\n")

# ===== Full optimizer with differential LR =====
backbone_params = list(model.encoder.parameters())
backbone_ids = set(id(p) for p in backbone_params)
head_params = [p for p in model.parameters() if id(p) not in backbone_ids]
loss_learnable = list(loss_fn.parameters())
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": cfg.lr_backbone},
    {"params": head_params, "lr": cfg.lr_head},
    {"params": loss_learnable, "lr": cfg.lr_loss_weights},  # Slow LR for DABL-C ratio
])

total_epochs_all = cfg.epochs_stage1 + cfg.epochs_stage2 + cfg.epochs_stage3
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_epochs_all, eta_min=1e-7
)

print("Stage 1: GT training (labels rotated each epoch)")
best_biou, best_state = run_stage(
    model, optimizer, scheduler, loss_fn,
    val_loader, cfg.epochs_stage1, "S1", best_biou, best_state)

# ===== Stage 2: FREEZE backbone to protect ViT features from noisy pseudo-labels =====
print("\nStage 2: Pseudo-label + rotated GT (backbone FROZEN, 7-gate + erosion)")
if cfg.pseudo_freeze_backbone:
    for p in model.encoder.parameters():
        p.requires_grad = False
    # Rebuild optimizer with only head + loss params (backbone frozen)
    head_params_s2 = [p for p in model.parameters() if p.requires_grad]
    optimizer_s2 = torch.optim.AdamW([
        {"params": head_params_s2, "lr": cfg.lr_head * 0.3},   # Lower LR for pseudo
        {"params": loss_learnable, "lr": cfg.lr_loss_weights},
    ])
    scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_s2, T_max=cfg.epochs_stage2, eta_min=1e-7
    )
    best_biou, best_state = run_stage(
        model, optimizer_s2, scheduler_s2, loss_fn,
        val_loader, cfg.epochs_stage2, "S2", best_biou, best_state,
        use_pseudo=True)
    # Unfreeze backbone for Stage 3
    for p in model.encoder.parameters():
        p.requires_grad = True
else:
    best_biou, best_state = run_stage(
        model, optimizer, scheduler, loss_fn,
        val_loader, cfg.epochs_stage2, "S2", best_biou, best_state,
        use_pseudo=True)

# ===== Stage 3: Full finetune (backbone unfrozen again) =====
print("\nStage 3: Finetune (labels rotated, full model)")
# Rebuild full optimizer for Stage 3
backbone_params_s3 = list(model.encoder.parameters())
backbone_ids_s3 = set(id(p) for p in backbone_params_s3)
head_params_s3 = [p for p in model.parameters() if id(p) not in backbone_ids_s3]
optimizer_s3 = torch.optim.AdamW([
    {"params": backbone_params_s3, "lr": cfg.lr_backbone * 0.5},  # Gentler backbone LR for finetune
    {"params": head_params_s3, "lr": cfg.lr_head * 0.5},
    {"params": loss_learnable, "lr": cfg.lr_loss_weights},
])
scheduler_s3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_s3, T_max=cfg.epochs_stage3, eta_min=1e-7
)
best_biou, best_state = run_stage(
    model, optimizer_s3, scheduler_s3, loss_fn,
    val_loader, cfg.epochs_stage3, "S3", best_biou, best_state)

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model (biou={best_biou:.4f}) from {cfg.save_path}")

# ──────── Plot DABL-C weight evolution (paper figure) ────────
if weight_history["epoch"]:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    epochs = weight_history["epoch"]
    ax.plot(epochs, weight_history["w_in"], 'b-o', markersize=3, label='w_in (building)')
    ax.plot(epochs, weight_history["w_out"], 'r-s', markersize=3, label='w_out (background)')
    stages = weight_history["stage"]
    prev_stage, start = stages[0], epochs[0]
    for i, (e, s) in enumerate(zip(epochs, stages)):
        if s != prev_stage:
            ax.axvline(x=e, color='gray', linestyle='--', alpha=0.5)
            ax.text(start + (e - start) / 2, ax.get_ylim()[1] * 0.95, prev_stage,
                    ha='center', fontsize=9, color='gray')
            prev_stage, start = s, e
    ax.text(start + (epochs[-1] - start) / 2, ax.get_ylim()[1] * 0.95, prev_stage,
            ha='center', fontsize=9, color='gray')

<!-- Cell 11 -->
### Prediction preview (after training)
Runs one batch from the validation set and shows input, GT, and prediction.

In [ ]:
# Cell 12 — Load best model and preview (random samples + polygon refinement)
import matplotlib.pyplot as plt

if os.path.exists(cfg.save_path):
    model.load_state_dict(torch.load(cfg.save_path, map_location=device))
    print(f"Loaded best model from {cfg.save_path}")

model.eval()
dataset = val_loader.dataset
indices = random.sample(range(len(dataset)), min(4, len(dataset)))
for idx in indices:
    img_t, mask_t = dataset[idx]
    img_t = img_t.unsqueeze(0).to(device)
    mask_t = mask_t.unsqueeze(0).to(device)

    with torch.no_grad():
        probs = torch.sigmoid(model(img_t)["logits"])

    img_np  = img_t[0].permute(1, 2, 0).cpu().numpy()
    gt_np   = mask_t[0, 0].cpu().numpy()
    prob_np = probs[0, 0].cpu().numpy()
    raw_pred = (prob_np > cfg.pred_thr).astype(np.float32)
    _, polygon = refine_prediction(img_np, prob_np, thr=cfg.pred_thr)

    print(f"prob stats: min={prob_np.min():.6f} mean={prob_np.mean():.4f} max={prob_np.max():.6f}")
    fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
    for ax, title, data, cmap in zip(
        axes,
        ["Image", "GT", "Prob", f"Raw@{cfg.pred_thr}", "Polygon"],
        [img_np, gt_np, prob_np, raw_pred, polygon],
        [None, "gray", "magma", "gray", "gray"]
    ):
        ax.set_title(title, fontsize=11); ax.imshow(data, cmap=cmap); ax.axis("off")
    plt.tight_layout(); plt.show()


<!-- Cell 13 -->
### Sanity check (labels)
Quick check to confirm masks are non-empty and aligned with images.

In [ ]:
# Cell 14
import matplotlib.pyplot as plt

imgs, masks = next(iter(labeled_loader))
print("mask mean:", masks.mean().item(), "min:", masks.min().item(), "max:", masks.max().item())

img0 = imgs[0].permute(1, 2, 0).numpy()
mask0 = masks[0, 0].numpy()

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.title("Image")
plt.imshow(img0)
plt.axis("off")
plt.subplot(1, 2, 2)
plt.title("Mask")
plt.imshow(mask0, cmap="gray")
plt.axis("off")
plt.show()


### Test Set Evaluation (Full Metrics)
Run the trained model on the held-out test split with all metrics:
IoU, Boundary-IoU (dilation), Hausdorff-95, ASSD, ECE.

In [ ]:
# Cell 16 — Test Set Evaluation (full metrics on held-out test split)
import matplotlib.pyplot as plt

def evaluate_full(model, loader, desc="eval"):
    """Evaluate with ALL metrics: IoU, Dice/F1, Precision, Recall, BIoU, HD95, ASSD, ECE."""
    model.eval()
    metrics = {"biou": [], "iou": [], "dice": [], "precision": [], "recall": [],
               "hd95": [], "assd": [], "ece": []}
    eps = 1e-6
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc=desc, dynamic_ncols=True):
            imgs, masks = imgs.to(device), masks.to(device)
            probs = torch.sigmoid(model(imgs)["logits"])
            pred_bin = (probs >= 0.5).float()

            # IoU
            metrics["biou"].append(boundary_iou(probs, masks).item())
            metrics["iou"].append(standard_iou(probs, masks).item())

            # Dice / F1
            inter = (pred_bin * masks).sum(dim=(1, 2, 3))
            pred_sum = pred_bin.sum(dim=(1, 2, 3))
            gt_sum = masks.sum(dim=(1, 2, 3))
            dice = ((2.0 * inter + eps) / (pred_sum + gt_sum + eps)).mean()
            metrics["dice"].append(dice.item())

            # Precision & Recall
            tp = inter  # true positives
            precision = ((tp + eps) / (pred_sum + eps)).mean()
            recall = ((tp + eps) / (gt_sum + eps)).mean()
            metrics["precision"].append(precision.item())
            metrics["recall"].append(recall.item())

            # Distance & calibration metrics
            metrics["hd95"].append(hausdorff_distance(probs, masks).item())
            metrics["assd"].append(assd_distance(probs, masks).item())
            metrics["ece"].append(ece_score(probs, masks).item())
    return {k: float(np.mean(v)) for k, v in metrics.items()}


# Build test loader
test_img_paths = load_images(TEST_IMG)
test_mask_paths = load_images(TEST_MASK)
n_test = min(len(test_img_paths), len(test_mask_paths))
test_img_paths = test_img_paths[:n_test]
test_mask_paths = test_mask_paths[:n_test]
print(f"Test set: {n_test} images")

if n_test > 0:
    test_ds = WHUDataset(test_img_paths, test_mask_paths, img_size=cfg.img_size, augment=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

    # Load best model
    if os.path.exists(cfg.save_path):
        model.load_state_dict(torch.load(cfg.save_path, map_location=device))
        print(f"Loaded best model from {cfg.save_path}")

    test_metrics = evaluate_full(model, test_loader, desc="Test")
    print("\n" + "="*55)
    print("TEST SET RESULTS")
    print("="*55)
    print(f"  IoU          : {test_metrics['iou']:.4f}")
    print(f"  Dice / F1    : {test_metrics['dice']:.4f}")
    print(f"  Precision    : {test_metrics['precision']:.4f}")
    print(f"  Recall       : {test_metrics['recall']:.4f}")
    print(f"  Boundary-IoU : {test_metrics['biou']:.4f}")
    print(f"  Hausdorff-95 : {test_metrics['hd95']:.2f} px")
    print(f"  ASSD         : {test_metrics['assd']:.2f} px")
    print(f"  ECE          : {test_metrics['ece']:.4f}")
    print("="*55)

    # Also evaluate on validation for comparison
    val_metrics = evaluate_full(model, val_loader, desc="Val")
    print("\nVALIDATION SET RESULTS")
    print("="*55)
    print(f"  IoU          : {val_metrics['iou']:.4f}")
    print(f"  Dice / F1    : {val_metrics['dice']:.4f}")
    print(f"  Precision    : {val_metrics['precision']:.4f}")
    print(f"  Recall       : {val_metrics['recall']:.4f}")
    print(f"  Boundary-IoU : {val_metrics['biou']:.4f}")
    print(f"  Hausdorff-95 : {val_metrics['hd95']:.2f} px")
    print(f"  ASSD         : {val_metrics['assd']:.2f} px")
    print(f"  ECE          : {val_metrics['ece']:.4f}")
    print("="*55)
else:
    print("No test images found — skipping test evaluation.")

## Confidence Map Visualization
Shows the learned confidence map alongside predictions.  
Key paper figure: **"The confidence head learns to be uncertain at building boundaries."**

In [ ]:
# Cell — Uncertainty Visualization (EDL + CLAAM + Confidence) — paper figures
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def visualize_uncertainty(model, loader, samples=4):
    """Show comprehensive uncertainty visualization:
    Image | GT | Prediction | EDL Uncertainty | CLAAM Agreement | Confidence Overlay
    Demonstrates that EDL captures epistemic uncertainty at boundaries,
    and CLAAM reveals where ViT layers disagree (encoder-level uncertainty)."""
    model.eval()
    dataset = loader.dataset
    n = len(dataset)
    indices = random.sample(range(n), min(samples, n))
    imgs_list, masks_list = [], []
    for idx in indices:
        item = dataset[idx]
        imgs_list.append(item[0])
        masks_list.append(item[1])
    imgs = torch.stack(imgs_list).to(device)
    masks = torch.stack(masks_list).to(device)

    with torch.no_grad():
        out = model(imgs)
        probs = torch.sigmoid(out["logits"])
        conf = out.get("conf")
        edl_out = out.get("edl")
        claam_out = out.get("claam")

    has_edl = edl_out is not None
    has_claam = claam_out is not None
    ncols = 4 + int(has_edl) + int(has_claam) + int(conf is not None) + int(has_claam)

    fig, axes = plt.subplots(samples, ncols, figsize=(4.5 * ncols, 4 * samples))
    if samples == 1:
        axes = axes[np.newaxis, :]

    for i in range(samples):
        img_np = imgs[i].permute(1, 2, 0).cpu().numpy()
        gt_np = masks[i, 0].cpu().numpy()
        prob_np = probs[i, 0].cpu().numpy()
        pred_np = (prob_np > cfg.pred_thr).astype(np.float32)

        col = 0
        # Col 0: Image
        axes[i, col].imshow(img_np)
        axes[i, col].set_title("Image", fontsize=11)
        col += 1

        # Col 1: Ground Truth
        axes[i, col].imshow(gt_np, cmap="gray")
        axes[i, col].set_title("Ground Truth", fontsize=11)
        col += 1

        # Col 2: Prediction
        axes[i, col].imshow(pred_np, cmap="gray")
        axes[i, col].set_title(f"Prediction @{cfg.pred_thr}", fontsize=11)
        col += 1

        # Col 3: Probability map
        im = axes[i, col].imshow(prob_np, cmap="magma", vmin=0, vmax=1)
        axes[i, col].set_title("Probability", fontsize=11)
        plt.colorbar(im, ax=axes[i, col], fraction=0.046, pad=0.04)
        col += 1

        # Col 4 (optional): EDL Epistemic Uncertainty
        if has_edl:
            edl_unc = edl_out["uncertainty"][i, 0].cpu().numpy()
            im_edl = axes[i, col].imshow(edl_unc, cmap="hot", vmin=0, vmax=1.0)
            axes[i, col].set_title("EDL Uncertainty", fontsize=11)
            plt.colorbar(im_edl, ax=axes[i, col], fraction=0.046, pad=0.04)
            col += 1

        # Col 5 (optional): CLAAM Agreement Map
        if has_claam:
            claam_agree = claam_out["agreement"][i, 0].cpu().numpy()
            im_claam = axes[i, col].imshow(claam_agree, cmap="RdYlGn", vmin=0, vmax=1)
            axes[i, col].set_title("CLAAM Agreement", fontsize=11)
            plt.colorbar(im_claam, ax=axes[i, col], fraction=0.046, pad=0.04)
            col += 1

        # Col 6 (optional): CLAAM layer variance
        if has_claam:
            claam_var = claam_out["variance"][i, 0].cpu().numpy()
            im_var = axes[i, col].imshow(claam_var, cmap="hot", vmin=0, vmax=0.25)
            axes[i, col].set_title("CLAAM Variance", fontsize=11)
            plt.colorbar(im_var, ax=axes[i, col], fraction=0.046, pad=0.04)
            col += 1

        # Col 7 (optional): Confidence
        if conf is not None:
            conf_np = conf[i, 0].cpu().numpy()
            im_conf = axes[i, col].imshow(conf_np, cmap="RdYlGn", vmin=0, vmax=1)
            axes[i, col].set_title("Confidence" + (" (EDL)" if has_edl else " (sigmoid)"),
                                   fontsize=11)
            plt.colorbar(im_conf, ax=axes[i, col], fraction=0.046, pad=0.04)
            col += 1

        for j in range(ncols):
            axes[i, j].axis("off")

    title_parts = ["Uncertainty Analysis"]
    if has_edl: title_parts.append("EDL Dirichlet")
    if has_claam: title_parts.append("CLAAM/MLAA")
    plt.suptitle(" + ".join(title_parts), fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    # ── Print detailed statistics ──
    print(f"\n{'='*60}")
    print(f"Uncertainty Statistics ({samples} samples)")
    print(f"{'='*60}")
    if has_edl:
        edl_unc_all = edl_out["uncertainty"].cpu().numpy()
        print(f"  EDL epistemic uncertainty: mean={edl_unc_all.mean():.4f}, "
              f"std={edl_unc_all.std():.4f}, max={edl_unc_all.max():.4f}")
        edl_strength = edl_out["strength"].cpu().numpy()
        print(f"  EDL Dirichlet strength:    mean={edl_strength.mean():.2f}, "
              f"min={edl_strength.min():.2f}, max={edl_strength.max():.2f}")
    if has_claam:
        agree_all = claam_out["agreement"].cpu().numpy()
        var_all = claam_out["variance"].cpu().numpy()
        print(f"  CLAAM agreement: mean={agree_all.mean():.4f}, "
              f"std={agree_all.std():.4f}, min={agree_all.min():.4f}")
        print(f"  CLAAM variance:  mean={var_all.mean():.6f}, max={var_all.max():.6f}")
    if conf is not None:
        conf_all = conf.cpu().numpy()
        print(f"  Confidence:      mean={conf_all.mean():.4f}, "
              f"std={conf_all.std():.4f}, min={conf_all.min():.4f}")

    # ── Boundary vs Interior analysis ──
    from scipy.ndimage import binary_dilation, binary_erosion
    for i in range(min(2, samples)):
        gt = masks[i, 0].cpu().numpy().astype(bool)
        struct = np.ones((7, 7))
        boundary = gt ^ binary_erosion(gt, structure=struct)
        interior = binary_erosion(gt, structure=struct)
        bg = ~binary_dilation(gt, structure=struct)
        if boundary.any() and interior.any():
            print(f"\n  Sample {i} — Boundary vs Interior:")
            if has_edl:
                u = edl_out["uncertainty"][i, 0].cpu().numpy()
                print(f"    EDL unc:   boundary={u[boundary].mean():.4f}  "
                      f"interior={u[interior].mean():.4f}  bg={u[bg].mean():.4f}")
            if has_claam:
                a = claam_out["agreement"][i, 0].cpu().numpy()
                print(f"    CLAAM agr: boundary={a[boundary].mean():.4f}  "
                      f"interior={a[interior].mean():.4f}  bg={a[bg].mean():.4f}")
            if conf is not None:
                c = conf[i, 0].cpu().numpy()
                print(f"    Confidence: boundary={c[boundary].mean():.4f}  "
                      f"interior={c[interior].mean():.4f}  bg={c[bg].mean():.4f}")


# --- Run uncertainty visualization ---
if os.path.exists(cfg.save_path):
    model.load_state_dict(torch.load(cfg.save_path, map_location=device))
    print(f"Loaded best model from {cfg.save_path}")
visualize_uncertainty(model, val_loader, samples=4)

### Cross-Dataset Evaluation (Inria Aerial)
Test the WHU-trained model on Inria Aerial Image Labeling dataset to measure generalization.
Inria images are 5000×5000 — we tile them into 512×512 patches for inference.
Dataset: [Inria Aerial Image Labeling](https://project.inria.fr/aerialimagelabeling/)
Upload to Kaggle as a dataset and set `cfg.inria_root` accordingly.

In [ ]:
# Cell 20 — 7-Gate Visualization + CLAAM inversion test
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch.nn.functional as F

# ── CONFIG ──────────────────────────────────────────────────────────────────
CHECKPOINT = "best_model.pt"